In [175]:
import sys
sys.path.append("../..")
import importlib
import clean_data

importlib.reload(clean_data)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path 
from catboost import CatBoostClassifier, Pool 

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, log_loss, f1_score, balanced_accuracy_score, accuracy_score

from clean_data import feature_engineering_days

In [176]:
df = pd.read_csv("../../data/austin/raw/aac_intakes.csv")
outcomes = pd.read_csv("../../data/austin/raw/aac_intakes_outcomes.csv")

df["datetime"] = pd.to_datetime(df["datetime"])
outcomes["intake_datetime"] = pd.to_datetime(outcomes["intake_datetime"])

df = df.merge(
    outcomes[["animal_id_intake", "intake_datetime", "time_in_shelter_days", "outcome_type"]],
    left_on=["animal_id", "datetime"],
    right_on=["animal_id_intake", "intake_datetime"],
    how="inner"
)

df = df[df["animal_type"] == "Cat"].copy()

In [177]:
pd.crosstab(
    df["intake_type"],
    df["outcome_type"]
)

outcome_type,Adoption,Died,Disposal,Euthanasia,Missing,Return to Owner,Rto-Adopt,Transfer
intake_type,,,,,,,,
Euthanasia Request,9,0,0,38,0,1,0,10
Owner Surrender,3719,60,0,126,6,225,15,1608
Public Assist,135,7,0,19,0,287,0,223
Stray,8803,334,13,1284,23,952,21,11461


In [178]:
pd.crosstab(
    df["intake_condition"],
    df["outcome_type"]
)

outcome_type,Adoption,Died,Disposal,Euthanasia,Missing,Return to Owner,Rto-Adopt,Transfer
intake_condition,,,,,,,,
Aged,10,1,0,14,0,5,0,9
Feral,15,0,0,4,0,5,0,59
Injured,380,64,2,543,0,120,6,386
Normal,11737,251,10,529,26,1269,27,11465
Nursing,245,44,1,21,2,0,0,948
Other,8,0,0,4,0,3,0,27
Pregnant,7,1,0,0,0,1,0,6
Sick,264,40,0,352,1,62,3,402


In [179]:
df = df[df["outcome_type"].isin(["Adoption", "Transfer"])].copy()

In [180]:
pd.crosstab(
    df["outcome_type"],
    pd.cut(
        df["time_in_shelter_days"],
        bins=[-float("inf"), 4, 10, float("inf")],
        labels=["<=4", "4-10", ">10"]
    ),
    normalize="index"
).mul(100).round(2)

time_in_shelter_days,<=4,4-10,>10
outcome_type,,,
Adoption,7.45,24.64,67.91
Transfer,61.63,20.56,17.81


In [181]:
df = feature_engineering_days(df)

df_cats = df[df["animal_type"] == "Cat"].copy()
df_cats = df_cats.sort_values("intake_datetime").reset_index(drop=True)

bins = [-np.inf, 4, 10, np.inf]
df_cats["target"] = pd.cut(
    df_cats["time_in_shelter_days"], bins=bins, labels=[0, 1, 2]
).astype(int)

In [182]:
drop_cols = [
    "target", "time_in_shelter_days", "time_in_shelter",
    "outcome_type", "outcome_subtype", "sex_upon_outcome",
    "intake_datetime", "animal_type", "young_and_cat", "datetime", "animal_id_intake", "outcome_type"
]

X = df_cats.drop(columns=drop_cols, errors="ignore")
y = df_cats["target"]

#categorical features which will be available
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
X[cat_features] = X[cat_features].fillna("Unknown").astype(str)

#70/15/15
n = len(X)
i1, i2 = int(n * 0.70), int(n * 0.85)

X_train, X_val, X_test = X.iloc[:i1], X.iloc[i1:i2], X.iloc[i2:]
y_train, y_val, y_test = y.iloc[:i1], y.iloc[i1:i2], y.iloc[i2:]

print("Split:", len(X_train), len(X_val), len(X_test))

Split: 18164 3892 3893


In [183]:
model = CatBoostClassifier(
    iterations=1200, learning_rate=0.03, depth=6,
    loss_function="MultiClass", random_seed=42, verbose=100
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    use_best_model=True
)

0:	learn: 1.0777772	test: 1.0820352	best: 1.0820352 (0)	total: 63.8ms	remaining: 1m 16s
100:	learn: 0.6885121	test: 0.7818951	best: 0.7815128 (99)	total: 6.55s	remaining: 1m 11s
200:	learn: 0.6615710	test: 0.7814930	best: 0.7798574 (132)	total: 13.1s	remaining: 1m 4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7798573981
bestIteration = 132

Shrink model to first 133 iterations.


CatBoostClassifier(depth=6, iterations=1200, learning_rate=0.03, loss_function='MultiClass', random_seed=42, verbose=100)

In [184]:
X_train.columns

Index(['breed', 'color', 'intake_condition', 'intake_type', 'name',
       'intake_hour', 'month_sin', 'month_cos', 'sex', 'intact',
       'is_mixed_breed', 'primary_breed', 'is_mixed_color', 'primary_color',
       'has_name', 'name_length', 'age_days', 'day_of_week', 'month', 'year',
       'is_weekend', 'is_puppy', 'is_senior', 'age_log', 'puppy_and_intact',
       'stray_and_intact', 'breed_age'],
      dtype='object')

# Evaluation


In [185]:
probs = model.predict_proba(X_test)
y_pred = model.classes_[np.argmax(probs, axis=1)]

#Log Loss
ll = log_loss(y_test, probs, labels=model.classes_)

#Multiclass Brier Score
y_onehot = (
    np.asarray(y_test)[:, None]
    == np.asarray(model.classes_)[None, :]
).astype(int)

brier = np.mean(np.sum((probs - y_onehot) ** 2, axis=1))

#Macro F1
f1 = f1_score(y_test, y_pred, average="macro")

#Balanced Accuracy
bal_acc = balanced_accuracy_score(y_test, y_pred)

print(f"Log Loss:          {ll:.4f}")
print(f"Brier Score:       {brier:.4f}")
print(f"Macro F1:          {f1:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Log Loss:          0.8662
Brier Score:       0.5221
Macro F1:          0.5385
Balanced Accuracy: 0.5662

Classification report:
              precision    recall  f1-score   support

           0       0.48      0.61      0.54       972
           1       0.60      0.24      0.34      1281
           2       0.65      0.85      0.73      1640

    accuracy                           0.59      3893
   macro avg       0.58      0.57      0.54      3893
weighted avg       0.59      0.59      0.56      3893



In [186]:
from sklearn.metrics import precision_recall_curve, average_precision_score

classes = list(model.classes_)
i2 = classes.index(2)

p2_val = model.predict_proba(X_val)[:, i2]
p2_test = model.predict_proba(X_test)[:, i2]

# Wybór progu na walidacji: najwyższa precyzja przy recall >= 0.6
prec, rec, thr = precision_recall_curve((y_val == 2).astype(int), p2_val)
ok = rec[:-1] >= 0.8
threshold = thr[ok][np.argmax(prec[:-1][ok])]
print(f"Próg dla klasy >30 dni: {threshold:.3f}")

# Ocena na teście
flag = p2_test >= threshold
y_long = (y_test == 2).astype(int)
tp = np.sum(flag & (y_long == 1))
print(f"Oznaczone: {flag.mean():.1%} zwierząt")
print(f"Precyzja: {tp / max(flag.sum(), 1):.3f}")
print(f"Recall:   {tp / y_long.sum():.3f}")
print(f"PR-AUC (klasa >30): {average_precision_score(y_long, p2_test):.3f}")

Próg dla klasy >30 dni: 0.524
Oznaczone: 47.6% zwierząt
Precyzja: 0.708
Recall:   0.801
PR-AUC (klasa >30): 0.754


In [187]:
def predict_cat(model, cat):
    probs = model.predict_proba(cat)[0]

    return pd.DataFrame({
        "Time in shelter": ["≤4 days", "4–10 days", ">30 days"],
        "Probability (%)": [
            round(probs[list(model.classes_).index(i)] * 100, 2)
            for i in [0, 1, 2]
        ]
    })


In [188]:
cat = X_test.iloc[[0]]
print(df_cats.loc[cat.index[0], "time_in_shelter_days"])
display(predict_cat(model, cat))

0.1555555555555555


,Time in shelter,Probability (%)
0,≤4 days,40.23
1,4–10 days,22.36
2,>30 days,37.41


In [189]:
cat = X_test.iloc[[30]]

print(df_cats.loc[cat.index[0], "time_in_shelter_days"])

display(predict_cat(model, cat))

2.126388888888889


,Time in shelter,Probability (%)
0,≤4 days,7.95
1,4–10 days,33.31
2,>30 days,58.74


In [190]:
long_stay = X_test[df_cats.loc[X_test.index, "time_in_shelter_days"] > 60]

cat = long_stay.iloc[[1]]

print(df_cats.loc[cat.index[0], "time_in_shelter_days"])
display(predict_cat(model, cat))


64.07638888888889


,Time in shelter,Probability (%)
0,≤4 days,10.04
1,4–10 days,26.28
2,>30 days,63.68
